In [18]:
import matplotlib.pyplot as plt
import json
import os
import re

In [11]:
results_dir = "results/ModelEvaluation"

In [12]:
files = dict()
files["pretrain"] = []
files["cpt"] = []

for file in os.listdir(results_dir):
    if "CPT" in file:
        files["cpt"].append(os.path.join(results_dir, file))
    else:
        files["pretrain"].append(os.path.join(results_dir, file))

In [50]:
import re

def scientific_to_decimal(s):
    """Convert a sci-notation string like '5e-2', '5e-2-', '5e-2-e', or '5e-2-e-' to a decimal string '0.05'."""
    if s == "unknown":
        return s
    # First, remove any '-e' or '-e-' at the end (to cover e.g. '5e-2-e' and '5e-2-e-')
    s = re.sub(r'(-e-?|(-e)?-)$', '', s)
    # Remove trailing non-alphanumerics (e.g. '-' or other chars after number)
    s = re.sub(r'[^0-9eE\+\-\.]+$', '', s)
    while s.endswith('-'):
        s = s[:-1]
    try:
        return str(float(s))
    except Exception:
        return s  # Return as is if conversion fails

def extract_pretrain_info(filename):
    d = {}
    if "sam" in filename:
        d["type"] = "sam"
        m = re.search(r"-rho([0-9eE\+\-\.e\-]*)", filename)
        if m:
            d["rho"] = scientific_to_decimal(m.group(1))
        else:
            d["rho"] = "unknown"
    elif "sgd" in filename:
        d["type"] = "sgd"
    else:
        d["type"] = "unknown"
    return d

def extract_cpt_info(filename):
    d = {}
    if "sam" in filename:
        d["type"] = "sam"
        m = re.search(r"-rho([0-9eE\+\-\.e\-]*)", filename)
        if m:
            d["rho"] = scientific_to_decimal(m.group(1))
        else:
            d["rho"] = "unknown"
    elif "sgd" in filename:
        d["type"] = "sgd"
    else:
        d["type"] = "unknown"
    m = re.search(r"-CPT-starcoder-lr([0-9eE\+\-\.e\-]*)", filename)
    if m:
        d["lr"] = scientific_to_decimal(m.group(1))
    else:
        d["lr"] = "unknown"
    return d

def find_metric_value(data, name):
    # Check for keys containing the task name, e.g. "c4" or "starcoder"
    for k, v in data.items():
        if name in k:
            # round to 3 decimal places if value is a float
            if isinstance(v, float):
                return round(v, 3)
            else:
                return v
    return None

results_by_group = {"pretrain": {}, "cpt": {}}

for group in ["pretrain", "cpt"]:
    for file in files[group]:
        with open(file, "r") as f:
            data = json.load(f)
        bn = os.path.basename(file)
        if group == "pretrain":
            info = extract_pretrain_info(bn)
            if info["type"] == "sam":
                rho = info["rho"]
                if "sam" not in results_by_group["pretrain"]:
                    results_by_group["pretrain"]["sam"] = {}
                if rho not in results_by_group["pretrain"]["sam"]:
                    results_by_group["pretrain"]["sam"][rho] = {}
                results_by_group["pretrain"]["sam"][rho]["c4"] = find_metric_value(data, "c4")
                results_by_group["pretrain"]["sam"][rho]["starcoder"] = find_metric_value(data, "starcoder")
            elif info["type"] == "sgd":
                if "sgd" not in results_by_group["pretrain"]:
                    results_by_group["pretrain"]["sgd"] = {}
                results_by_group["pretrain"]["sgd"]["c4"] = find_metric_value(data, "c4")
                results_by_group["pretrain"]["sgd"]["starcoder"] = find_metric_value(data, "starcoder")
        elif group == "cpt":
            info = extract_cpt_info(bn)
            if info["type"] == "sam":
                rho = info["rho"]
                lr = info["lr"]
                if "sam" not in results_by_group["cpt"]:
                    results_by_group["cpt"]["sam"] = {}
                if rho not in results_by_group["cpt"]["sam"]:
                    results_by_group["cpt"]["sam"][rho] = {}
                if lr not in results_by_group["cpt"]["sam"][rho]:
                    results_by_group["cpt"]["sam"][rho][lr] = {}
                results_by_group["cpt"]["sam"][rho][lr]["c4"] = find_metric_value(data, "c4")
                results_by_group["cpt"]["sam"][rho][lr]["starcoder"] = find_metric_value(data, "starcoder")
            elif info["type"] == "sgd":
                lr = info["lr"]
                if "sgd" not in results_by_group["cpt"]:
                    results_by_group["cpt"]["sgd"] = {}
                if lr not in results_by_group["cpt"]["sgd"]:
                    results_by_group["cpt"]["sgd"][lr] = {}
                results_by_group["cpt"]["sgd"][lr]["c4"] = find_metric_value(data, "c4")
                results_by_group["cpt"]["sgd"][lr]["starcoder"] = find_metric_value(data, "starcoder")


In [52]:
results_by_group["cpt"]

{'sam': {'0.01': {'0.001': {'c4': 6.77, 'starcoder': 5.306},
   '0.0002': {'c4': 7.136, 'starcoder': 6.715},
   '4e-05': {'c4': 7.278, 'starcoder': 8.038}},
  '0.1': {'0.001': {'c4': 6.846, 'starcoder': 5.384},
   '4e-05': {'c4': 7.425, 'starcoder': 8.12},
   '0.0002': {'c4': 7.232, 'starcoder': 6.784}},
  '0.05': {'4e-05': {'c4': 7.334, 'starcoder': 8.085},
   '0.0002': {'c4': 7.164, 'starcoder': 6.763},
   '0.001': {'c4': 6.798, 'starcoder': 5.344}},
  '0.0': {'0.0002': {'c4': 7.126, 'starcoder': 6.706},
   '4e-05': {'c4': 7.268, 'starcoder': 8.029},
   '0.001': {'c4': 6.794, 'starcoder': 5.339}},
  '0.02': {'4e-05': {'c4': 7.291, 'starcoder': 8.049},
   '0.001': {'c4': 6.782, 'starcoder': 5.313},
   '0.0002': {'c4': 7.144, 'starcoder': 6.729}}},
 'sgd': {'0.001': {'c4': 6.794, 'starcoder': 5.339},
  '0.0002': {'c4': 7.126, 'starcoder': 6.706},
  '4e-05': {'c4': 7.268, 'starcoder': 8.029}}}

In [63]:
import matplotlib.pyplot as plt
import numpy as np
import os

# --------- Prepare results_dir/plots output folder ---------
plots_dir = os.path.join(results_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

# --------- Pretrain Plots (Side by Side for c4 and starcoder, SAM only, log scores) ---------

pretrain = results_by_group["pretrain"]
sam_pretrain = pretrain['sam']  # dict: rho -> {'c4': v, 'starcoder': v}

# Get sorted rho values (as float for better plotting)
rhos = sorted([float(rho) for rho in sam_pretrain.keys()])
rhos_str = [str(rho) for rho in rhos]

c4_scores = [sam_pretrain[str(rho)]['c4'] for rho in rhos]
sc_scores = [sam_pretrain[str(rho)]['starcoder'] for rho in rhos]

log_c4_scores = np.log(c4_scores)
log_sc_scores = np.log(sc_scores)

fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
# c4 subplot
axs[0].plot(rhos, log_c4_scores, marker='o', color='tab:blue')
axs[0].set_xlabel("rho")
axs[0].set_ylabel("Log Perplexity")
axs[0].set_title("Pretrain: Log Perplexity C4 vs Rho (SAM)")
axs[0].grid(True)

# starcoder subplot
axs[1].plot(rhos, log_sc_scores, marker='s', color='tab:orange')
axs[1].set_xlabel("rho")
axs[1].set_ylabel("Log Perplexity")
axs[1].set_title("Pretrain: Log Perplexity Starcoder vs Rho (SAM)")
axs[1].grid(True)

plt.tight_layout()
fname = os.path.join(plots_dir, "pretrain_log_perplexity_c4_and_starcoder_vs_rho_sam.png")
plt.savefig(fname)
plt.close(fig)

# # --------- CPT Plots (Side by Side for c4 and starcoder, one for each lr), and one big plot with all lr, SAM only, log scores ---------

cpt = results_by_group["cpt"]
sam_cpt = cpt['sam'] # dict: rho -> lr -> {'c4': v, 'starcoder': v}

# List all unique learning rates present (order them for plotting)
lrs = sorted({float(lr) for rhos in sam_cpt.values() for lr in rhos})
lrs_str = [str(lr) for lr in lrs]

# # Side by side individual plots for each lr
for lr in lrs_str:
    rhos_with_lr = []
    c4_scores = []
    sc_scores = []
    for rho in sam_cpt:
        if lr in sam_cpt[rho]:
            rhos_with_lr.append(float(rho))
            c4_scores.append(sam_cpt[rho][lr]['c4'])
            sc_scores.append(sam_cpt[rho][lr]['starcoder'])
    if rhos_with_lr:
        order = sorted(range(len(rhos_with_lr)), key=lambda i: rhos_with_lr[i])
        rhos_sorted = [rhos_with_lr[i] for i in order]
        c4_sorted = [c4_scores[i] for i in order]
        sc_sorted = [sc_scores[i] for i in order]
        log_c4_sorted = np.log(c4_sorted)
        log_sc_sorted = np.log(sc_sorted)
        fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
        # c4 subplot
        axs[0].plot(rhos_sorted, log_c4_sorted, marker='o', color='tab:blue')
        axs[0].set_xlabel("Rho")
        axs[0].set_ylabel("Log Perplexity")
        axs[0].set_title(f"CPT: Log Perplexity C4 vs Rho (SAM), lr={lr}")
        axs[0].legend()
        axs[0].grid(True)

        # starcoder subplot
        axs[1].plot(rhos_sorted, log_sc_sorted, marker='s', color='tab:orange')
        axs[1].set_xlabel("Rho")
        axs[1].set_ylabel("Log Perplexity")
        axs[1].set_title(f"CPT: Log Perplexity Starcoder vs Rho (SAM), lr={lr}")
        axs[1].legend()
        axs[1].grid(True)

        plt.tight_layout()
        fname = os.path.join(plots_dir, f"cpt_log_perplexity_c4_and_starcoder_vs_rho_sam_lr{lr}.png")
        plt.savefig(fname)
        plt.close(fig)

# --------- Single combined CPT Plot: All 3 lr for both c4 and starcoder, SAM only, log scores ---------

# c4
fig_c4, ax_c4 = plt.subplots(figsize=(8,5))
for idx, lr in enumerate(lrs_str):
    rhos_with_lr = []
    c4_scores = []
    for rho in sam_cpt:
        if lr in sam_cpt[rho]:
            rhos_with_lr.append(float(rho))
            c4_scores.append(sam_cpt[rho][lr]['c4'])
    if rhos_with_lr:
        order = sorted(range(len(rhos_with_lr)), key=lambda i: rhos_with_lr[i])
        rhos_sorted = [rhos_with_lr[i] for i in order]
        c4_sorted = [c4_scores[i] for i in order]
        log_c4_sorted = np.log(c4_sorted)
        ax_c4.plot(rhos_sorted, log_c4_sorted, marker='o', label=f'lr={lr}')
ax_c4.set_xlabel("Rho")
ax_c4.set_ylabel("Log Perplexity")
ax_c4.set_title("CPT: Log Perplexity C4 vs Rho for all lr (SAM)")
# ax_c4.legend()
ax_c4.grid(True)
plt.tight_layout()
fname_c4 = os.path.join(plots_dir, "cpt_log_perplexity_c4_vs_rho_all_lr_sam.png")
plt.savefig(fname_c4)
plt.close(fig_c4)

# starcoder
fig_sc, ax_sc = plt.subplots(figsize=(8,5))
for idx, lr in enumerate(lrs_str):
    rhos_with_lr = []
    sc_scores = []
    for rho in sam_cpt:
        if lr in sam_cpt[rho]:
            rhos_with_lr.append(float(rho))
            sc_scores.append(sam_cpt[rho][lr]['starcoder'])
    if rhos_with_lr:
        order = sorted(range(len(rhos_with_lr)), key=lambda i: rhos_with_lr[i])
        rhos_sorted = [rhos_with_lr[i] for i in order]
        sc_sorted = [sc_scores[i] for i in order]
        log_sc_sorted = np.log(sc_sorted)
        ax_sc.plot(rhos_sorted, log_sc_sorted, marker='s', label=f'lr={lr}')
ax_sc.set_xlabel("Rho")
ax_sc.set_ylabel("Log Perplexity")
ax_sc.set_title("CPT: Log Perplexity Starcoder vs Rho for all lr (SAM)")
ax_sc.legend()
ax_sc.grid(True)
plt.tight_layout()
fname_sc = os.path.join(plots_dir, "cpt_log_perplexity_starcoder_vs_rho_all_lr_sam.png")
plt.savefig(fname_sc)
plt.close(fig_sc)


/mnt/tmp/2719300/ipykernel_495235/922332008.py:76: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axs[0].legend()
/mnt/tmp/2719300/ipykernel_495235/922332008.py:84: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axs[1].legend()
